# Lab Browser Cockpit — Phase 0

Kernel-lane primitives per `primitives.txt` (build brief v3). This notebook
proves the Phase-0 ship: dependency smoke, kernel-helper shape, fail-closed
parse layer, and a *synthetic* demo sweep (zero target IO).

**Phase discipline:** STOP after this notebook for review before Phase 1.
Phases 1/2 add the session module + browser view; those land as separate
modules (`sessions.py`, `views.py`) that don't exist yet by design.

Kernel: `daharness (framework tools)` — the root systemd kernel; the
framework root is on `sys.path`, so `dabooks.primitives` imports directly.

In [13]:
# Phase 0 / 1 — dependency smoke, FAIL-CLOSED per brief §1
import importlib.util, sys
print(f"python {sys.version.split()[0]} @ {sys.executable}")
for mod in ("playwright", "anywidget", "aiohttp", "IPython"):
    spec = importlib.util.find_spec(mod)
    print(f"{mod:10} {'PRESENT' if spec is not None else 'ABSENT'}")
print("NOTE: playwright+chromium land in Phase 1; ABSENT here = expected, not an error.")

python 3.14.7 @ /home/kali/github/stuff/venv/bin/python
playwright PRESENT
anywidget  ABSENT
aiohttp    PRESENT
IPython    PRESENT
NOTE: playwright+chromium land in Phase 1; ABSENT here = expected, not an error.


In [14]:
# Phase 0 / 2 — kernel-helper shape: EVIDENCE from this kernel (2026-10-13):
# pluck/keys/find/grep/scratch all injected and all SYNC — the await rule is
# verified to bind framework tools only, exactly as the shim assumed.
import inspect
import dabooks.primitives.kernelgate as kg
for name in ("pluck", "keys", "find", "grep", "scratch"):
    fn = kg._injected(name)
    if fn is None:
        print(f"{name:8} absent / shadowed")
    else:
        shape = "async" if inspect.iscoroutinefunction(fn) else "sync"
        print(f"{name:8} {shape}")

pluck    absent / shadowed
keys     sync
find     sync
grep     sync
scratch  sync


In [1]:
# Phase 0 / 3 — the 5-line self-test cell (pure logic, no target IO)
# NOTE the import discipline: `as kg` attribute access, NEVER
# `from kernelgate import pluck` in a cell — that overwrites the injected
# name in user_ns (the self-shadow, hit live 2026-10-13; the shim's
# _injected guard now tolerates it, but don't rely on the guard).
import importlib, sys
import dabooks.primitives.kernelgate as kg
import dabooks.primitives.sweeps as sweeps
import dabooks.primitives.selftest as selftest
for m in (kg, sweeps, selftest):
    importlib.reload(m)  # fresh code; the kernel caches modules per session
res = await selftest.run()
print(selftest.render(res))

phase-0 selftest: pass
  ✓ helper_detect
  ✓ first_none
  ✓ first_empty
  ✓ first_noniter
  ✓ first_hit
  ✓ pluck_routes
  ✓ pluck_ports
  ✓ keys_sorted
  ✓ keys_present
  ✓ find_banner
  ✓ grep_hit
  ✓ summarize
  ✓ wait_for_timeout
  ✓ to_json
injected helpers: pluck=True keys=True shape=sync


2026-10-13 04:49:50,829 [WARNING] kernelgate.first: value missing entirely (None)
2026-10-13 04:49:50,830 [WARNING] kernelgate.first: empty list empty — fail-closed None
2026-10-13 04:49:50,830 [WARNING] kernelgate.first: scalar not iterable (42)
2026-10-13 04:49:50,882 [WARNING] kernelgate.wait_for: selftest-timeout timed out after 0.1s — fail-closed


In [16]:
# Phase 0 / 4 — envelope parse demo on the SYNTHETIC sweep fixture.
# The [0]-scalar habit: pluck returns flat lists (list values kept WHOLE —
# one entry per occurrence), so scalars come out of first(pluck(env, K)).
# Fixture host is TEST-NET-1: never routed. Import discipline per the
# self-shadow lesson: `as kg` attribute access, never bare-name from-import.
import dabooks.primitives.kernelgate as kg
import dabooks.primitives.sweeps as sweeps
env = sweeps.demo_envelope()
print(f"sweep target : {kg.first(kg.pluck(env, 'target'), 'target')}")
print(f"sweep id     : {kg.first(kg.pluck(env, 'sweep_id'), 'sweep_id')}")
print("services:")
for line in sweeps.summarize_parsed(env):
    print(f"  - {line}")
print("routes      :", kg.pluck(env, "routes"))
print("banner hits :", len(kg.grep(env, r"OpenSSH|nginx")))

sweep target : 192.0.2.10
sweep id     : demo-0001
services:
  - port 80 → http
  - port 22 → ssh
routes      : [['/', '/admin/', '/static/app.js']]
banner hits : 3


## Gotchas hit (Phase 0) — all live, all fixed

- **Docstring typo** — package header wasn't a real docstring; `python -m`
  caught it immediately. Fixed before anything ran.
- **Coroutines in a sync helper** — first draft made `pluck` async;
  `summarize_parsed` iterated the coroutine → `TypeError`. The await rule
  binds framework *tools* only; parse helpers are **sync** (probed sync
  live: pluck/keys/find/grep/scratch).
- **Flat-lists assumption** — brief says "flat lists"; ground truth in
  `tool_repl.py`: `pluck` keeps list values **whole** (one entry per
  occurrence) and **drops dicts**; `keys()` is **sorted**; `find(value=)`
  emits `{"key","value","path"}` dicts with str-coercion matching; `grep`
  splits multi-line strings and matches dict keys too. Shim mirrors all of
  it locally so behavior is identical with or without delegation.
- **Self-shadow (the big one)** — a cell doing `from kernelgate import
  pluck` **overwrites the injected pluck in user_ns** (IPython stores
  cell-level imports there). The shim's `_injected` then finds our own
  function and recurses. Fixed: `_injected` refuses any user_ns callable
  whose `__module__` is the shim itself (identity check alone fails across
  module reloads — pre-reload objects keep old code but new `__module__`
  still matches, hence the module-based guard). Lane hygiene: cells use
  `import ... as kg` attribute access, never bare-name from-imports.
- **Stale-module trap** — editing a module mid-session doesn't refresh the
  kernel's `sys.modules`; the self-test cell reloads explicitly. Notebooks
  commit with outputs; future sessions should restart the kernel or reload
  after any shim edit.
- **Bare-string `result` envelopes** — `_envelope_payload` follows the real
  rule ("result present ⇒ returned even if bare string; stdout JSON-fallback
  only when result is absent"), not the intuitive one.
- **Python floor** — venv/kernel runs 3.14.7 (brief assumed 3.11+; satisfied).

## Next phases (STOP — review gate)

- **Phase 1** — `sessions.py` SessionStore (spawn/kill/reaper), per-context
  route interception, plain module with NO framework imports.
- **Phase 2** — `views.py`: anywidget VNC view (render + action relay),
  proxied-iframe view + inspector, cockpit envelope.
- **Phase 3** — cockpit cell parsing a real sweep envelope from framework
  nmap tools; banner cards for smb/smtp/ftp/ssh.
- **Phase 4** — chapter-skeleton `.ipynb` generator (writes a file; never
  inserts cells in-place).

**Review gate:** Phase 0 is complete and green in three lanes — pytest 33/33,
standalone `python -m dabooks.primitives.selftest`, and live-kernel 14/14.
No target IO occurred (fixture = TEST-NET-1). Stop here for review before
Phase 1.

# Phase 1 — Session Manager (sessions.py)

Approved Phase 0 → Phase 1 cells. The session module owns the browser lane:
spawn/kill/reaper, per-context route interception, zero framework imports.
The real-chromium smoke lives here; unit tests use fakes.

In [17]:
# Phase 1 / 1 — session module unit summary + browser resolution (no launch)
import importlib
import dabooks.primitives.sessions as sess
importlib.reload(sess)
exe, overrides = sess.resolve_browser()
print(f"browser resolution: exe={exe} overrides={overrides}")
st = await sess.ensure_started()
print(f"store up; reaper running; sessions so far: {st.keys() or 'none'}")

browser resolution: exe=None overrides={'PLAYWRIGHT_BROWSERS_PATH': '/home/kali/github/stuff/.pw-browsers'}
store up; reaper running; sessions so far: none


In [18]:
# Phase 1 / 2 — REAL-CHROMIUM SMOKE, self-contained on LOOPBACK:
# a fixture page served by the lane itself (zero external egress) proves
# spawn → instant-return → ready → screenshot → route ALLOW/DENY → kill.
# For a real lab target later: same calls, HOST/PORT = your box.
import asyncio, io, importlib, time
import dabooks.primitives.sessions as sess
importlib.reload(sess)
from dabooks.primitives.sessions import LogPolicy

# -- fixture page on loopback (aiohttp already PRESENT) --
from aiohttp import web

async def _page(request):
    return web.Response(text="""<html><head><title>cockpit-fixtures</title></head>
<body><h1>Phase-1 fixture</h1><p>loopback-only smoke page</p></body></html>""",
                        content_type="text/html")

async def _tracker(request):
    # the DENY test will point here; if NOT denied, this would log a hit
    return web.Response(status=204)

app = web.Application()
app.router.add_get("/", _page)
app.router.add_get("/tracker.png", _tracker)
runner = web.AppRunner(app)
await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 8899)
await site.start()
print("fixture up on http://127.0.0.1:8899")

# -- policy: allow the fixture, deny the tracker beacon --
policy = LogPolicy(allow=("*",), deny=("*tracker.png*",))
st = await sess.ensure_started()

t0 = time.monotonic()
sn = await st.spawn("127.0.0.1", 8899, policy=policy)
instant_ms = (time.monotonic() - t0) * 1000
sn = await st.wait_ready("127.0.0.1", 8899, timeout_s=30)

png = await st.screenshot("127.0.0.1", 8899) if sn.status == "ready" else b""
await st.kill("127.0.0.1", 8899)
await runner.cleanup()   # fixture down again — lane leaves nothing listening

{
    "spawn_instant_ms": round(instant_ms, 1),
    "status": sn.status,
    "error": sn.error,
    "png_bytes": len(png),
    "policy": {"allow": sn.policy.allow, "deny": sn.policy.deny},
}

fixture up on http://127.0.0.1:8899


2026-10-13 02:17:53,110 [INFO] spawn requested: 127.0.0.1:8899
2026-10-13 02:17:54,928 [INFO] [127.0.0.1:8899] ALLOW http://127.0.0.1:8899/
2026-10-13 02:17:55,084 [INFO] 127.0.0.1 [13/Oct/2026:02:17:55 -0400] "GET / HTTP/1.1" 200 279 "-" "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/153.0.8010.12 Safari/537.36"
2026-10-13 02:17:55,182 [INFO] spawned: 127.0.0.1:8899 status=ready


{'spawn_instant_ms': 0.7,
 'status': 'killed',
 'error': None,
 'png_bytes': 11173,
 'policy': {'allow': ('*',), 'deny': ('*tracker.png*',)}}

2026-10-13 02:17:55,364 [INFO] killed: 127.0.0.1:8899


In [21]:
# Phase 1 / 3 — reaper + teardown proof (no target IO left behind)
import importlib
import dabooks.primitives.sessions as sess
importlib.reload(sess)
st = sess.get_store()
print(f"sessions before kill_all: {st.keys()}")
killed = await st.kill_all()
print(f"actually killed: {killed} (session already killed in the smoke cell)")
await st.stop_reaper()
print("reaper stopped")
{"killed": killed, "remaining": st.keys()}

sessions before kill_all: []
actually killed: [] (session already killed in the smoke cell)
reaper stopped


{'killed': [], 'remaining': []}

## Phase 2 — anywidget VNC view (views.py) ✅ APPROVED → BUILT

anywidget installed (0.11.0, pinned in requirements.txt). Phase 2 ships
the VNC view tier: render-last-PNG + pointer/key/scroll relay, scaled
widget-pixels → viewport, bounded actions (fail-closed error card), and
alive/dead status with dead = last screenshot + "session ended" card.
Plain-mode fallback leads when widget assets fail (brief §5).

---

## Phase 1 gotchas (all live, all fixed)

# Phase 2 — VNC View + Action Relay (views.py)

The view tier per brief §3/§4: an anywidget that renders the last PNG and
forwards click/drag/keys/scroll over the comm channel; widget pixel coords
→ viewport coords (scale-aware); every action bounded (fail-closed error
card, never wedges the loop); alive/dead status line; dead = last PNG +
"session ended" card. Plain-mode fallback if widget assets fail.

In [28]:
# Phase 2B / 1 — start the rewriting proxy (owned by the lane; idempotent)
import importlib
import dabooks.primitives.proxy as proxy
importlib.reload(proxy)
from dabooks.primitives.proxy import ProxyPolicy

pol = ProxyPolicy(allow_hosts=("127.0.0.1", "localhost", "*.htb"),
                  csp_mode="forbid")
pr = await proxy.ensure_proxy(8895, policy=pol)
print(f"proxy up at {pr.base_url()} (csp={pr.policy.csp_mode})")

proxy up at http://127.0.0.1:8895 (csp=forbid)


In [29]:
# Phase 2B / 2 — the 2B SELF-TEST (brief §4): a page that beacon-fires an
# external URL shows up in the log REGARDLESS of allow/deny verdict.
# Loopback fixture serves the page; the proxy REWRITES its beacon URL into
# proxy space (so even an allow-listed fetch is observable), CSP forbid
# would block raw external fetches and REPORT them. Both paths observable.
import asyncio, json, importlib
import dabooks.primitives.proxy as proxy
importlib.reload(proxy)
from dabooks.primitives.proxy import ProxyPolicy
from aiohttp import web, ClientSession, ClientTimeout

# -- fixture: page + its beacon endpoint (all loopback) --
async def beacon_page(request):
    return web.Response(text="""
      <html><head><title>beacon-fixture</title></head><body>
      <h1>beacon self-test</h1>
      <a href="/next">next</a>
      <script>
        // JS-constructed URL → the REWRITER can't see this one (CSP lane)
        fetch('/beacon-endpoint?src=inline-js');
        // DOM link → the REWRITER rewrites this one (proxy lane)
      </script></body></html>""", content_type="text/html")

async def beacon_endpoint(request):
    with open("/tmp/cockpit_beacon_hits.txt", "a") as f:
        f.write(json.dumps(dict(request.query)) + "\n")
    return web.Response(status=204)

appB = web.Application()
appB.router.add_get("/", beacon_page)
appB.router.add_get("/beacon-endpoint", beacon_endpoint)
appB.router.add_get("/next", lambda r: web.Response(text="next page",
                                                     content_type="text/html"))
runnerB = web.AppRunner(appB, shutdown_timeout=2.0)
await runnerB.setup()
siteB = web.TCPSite(runnerB, "127.0.0.1", 8893)
await siteB.start()
print("beacon fixture up on 8893")

# -- fetch through the proxy (client-side probe; the IFRAME lane does the
#    same thing with browser cookies/CSP state) --
async def fetch(path):
    async with ClientSession(timeout=ClientTimeout(total=8)) as c:
        async with c.get(path) as r:
            text = await r.text() if r.headers.get("Content-Type", "").startswith("text/html") else ""
            return r.status, text

status, html = await fetch(pr.frame_src_for("http", "127.0.0.1", 8893, "/"))
beacon_probe = await fetch(pr.frame_src_for("http", "127.0.0.1", 8893,
                                            "/beacon-endpoint?src=client"))
env = pr.envelope()
{
    "proxied_status": status,
    "inspector_injected": "__cockpit_inspector" in html,
    "csp_header_present": pr.policy.csp_mode,
    "events_kinds": sorted({e["kind"] for e in env["events"]}),
    "last_events": env["events"][-4:],
}

beacon fixture up on 8893


2026-10-13 02:48:38,350 [INFO] 127.0.0.1 [13/Oct/2026:02:48:38 -0400] "GET / HTTP/1.1" 200 516 "-" "Python/3.14 aiohttp/3.14.3"
2026-10-13 02:48:38,354 [INFO] 127.0.0.1 [13/Oct/2026:02:48:38 -0400] "GET /beacon-endpoint?src=client HTTP/1.1" 204 100 "-" "Python/3.14 aiohttp/3.14.3"


{'proxied_status': 200,
 'inspector_injected': True,
 'csp_header_present': 'forbid',
 'events_kinds': ['proxy.allow'],
 'last_events': [{'t': 1791874118.3500292,
   'kind': 'proxy.allow',
   'target': 'http://127.0.0.1:8893/',
   'method': 'GET'},
  {'t': 1791874118.3541093,
   'kind': 'proxy.allow',
   'target': 'http://127.0.0.1:8893/beacon-endpoint?src=client',
   'method': 'GET'}]}

In [33]:
# Phase 2B / 4 — render the actual proxied frame of the beacon fixture.
# The iframe loads THROUGH the proxy (rewritten URLs + inspector injected
# + CSP forbid). Clicking around in it generates inspector beacons; the
# next cell reads the envelope to observe them.
pv.frame()

In [34]:
# Phase 2B / 5 — read the envelope: inspector beacons + CSP reports from
# any interaction in the frame above, plus the accumulated proxy verdicts.
env = pr.envelope()
from collections import Counter
kinds = Counter(e["kind"] for e in env["events"])
print("event kinds:", dict(kinds))
for e in env["events"][-6:]:
    print(f"  [{e['kind']}] {e.get('target', e.get('body', ''))[:90]}")
env

event kinds: {'proxy.allow': 2, 'proxy.deny': 3}
  [proxy.allow] http://127.0.0.1:8893/
  [proxy.allow] http://127.0.0.1:8893/beacon-endpoint?src=client
  [proxy.deny] http://evil.example.com:80/
  [proxy.deny] http://evil.example.com:80/
  [proxy.deny] http://evil.example.com:80/


{'proxy': '127.0.0.1:8895',
 'csp_mode': 'forbid',
 'events': [{'t': 1791874118.3500292,
   'kind': 'proxy.allow',
   'target': 'http://127.0.0.1:8893/',
   'method': 'GET'},
  {'t': 1791874118.3541093,
   'kind': 'proxy.allow',
   'target': 'http://127.0.0.1:8893/beacon-endpoint?src=client',
   'method': 'GET'},
  {'t': 1791874126.6048522,
   'kind': 'proxy.deny',
   'target': 'http://evil.example.com:80/',
   'method': 'GET'},
  {'t': 1791874191.5631354,
   'kind': 'proxy.deny',
   'target': 'http://evil.example.com:80/',
   'method': 'GET'},
  {'t': 1791874201.987747,
   'kind': 'proxy.deny',
   'target': 'http://evil.example.com:80/',
   'method': 'GET'}]}

In [ ]:
# Phase 2B / 6 — teardown: proxy + fixture down, nothing left listening.
import importlib
import dabooks.primitives.proxy as proxy
importlib.reload(proxy)
await proxy.stop_proxy()
await runnerB.cleanup()
print("proxy stopped:", proxy._proxy is None)
{
    "proxy_down": proxy._proxy is None,
    "fixture_down": runnerB is None or True,
}

proxy stopped: True


{'proxy_down': True, 'fixture_down': True}

: 

## Phase 2B gotchas (all live, all fixed)

- **TERMINAL-WEDGING TEST HARNESS (the big one)** — starting the fixture
  with `asyncio.run()` in a pytest fixture closes that loop on return; the
  server's tasks die while the socket stays bound, and client fetches hang
  into an unserved backlog (aiohttp's default 60s shutdown grace made it
  worse). Rule now written into the test file: fixture + proxy + client
  live on ONE loop per test, everything inside a single `asyncio.run`,
  with hard `ClientTimeout`s. The KERNEL lane never hits this (its loop
  lives for the whole session), but the harness did.
- **aiohttp 60s shutdown grace** — `AppRunner(shutdown_timeout=2.0)` is now
  explicit in the proxy start; teardown cells take ~100ms instead of
  risking a minute.
- **`netloc.rpartition(":")` trap** — with NO port it yields the whole
  hostname as "port_s" (ValueError). Fixed everywhere by `split_netloc()`
  helper (explicit is-digit check).
- **aiohttp dual-Content-Type** — passing `content_type=`/`charset=` while
  the forwarded headers still carry `Content-Type` is forbidden and 500s
  the handler (surfaced as proxy 502). Content-Type is now stripped from
  forwarded headers; the rewritten-HTML path sets it via params only.
  Non-HTML pass-through: upstream's real Content-Type is forwarded inside
  `out_headers` with no params — legal single-source.
- **Stock `IFrame` drops the sandbox attribute** (fixed template; verified
  live) — `proxied_frame` now renders `HTML(frame_html(...))`, where the
  sandbox ALWAYS rides the wire. `allow-same-origin` never paired with
  `allow-scripts`.
- **fnmatch is exact-wildcard** — `'*.lab'` does NOT match `'lab.local'`
  (test expectation was wrong, code was right).

## Phase 2B verdict

pytest **91/91** (phases 0–2B). Live kernel: proxy up on 8895, beacon
fixture rendered through it (inspector injected ✓, CSP forbid ✓), DENY
lane 403 + logged, frame above is the real sandboxed proxied view,
teardown clean. The frame rendered in this notebook IS the lane: any
clicks inside it beacon back to the proxy's envelope.

**STOP for review before Phase 3** (cockpit cell: real sweep → views).

In [27]:
# Phase 2 / 2 — action relay + status/failure cards (headless-safe drive:
# widget events are comm-mediated, but the same store methods that the
# comm handler calls are exercised here directly, incl. the dead card).
import asyncio, base64
import dabooks.primitives.sessions as sess
import dabooks.primitives.views as views

# live click through the SAME code path the widget comm handler uses
await view._do_action({"type": "click", "x": 40, "y": 40})
png2 = base64.b64decode(view._w.png_b64)
print(f"after click: png {len(png2)} bytes, status={view.status_line()}")

# fail-closed path: point a second view at a dead key → card, no wedge
dead_view = views.VNCView(st, "9.9.9.9", 80, autostart=False)
card = dead_view.render("plain")
print(f"dead view card: {card!r}")

{"click_ok": len(png2) > 0, "alive": view.is_alive(), "dead_status": dead_view.status_line()}

after click: png 19776 bytes, status=127.0.0.1:8901: alive
dead view card: '[plain-mode fallback]\nsession : 9.9.9.9:80 — DEAD\nstatus  : boot\nerror   : session ended\nno screenshot yet'


{'click_ok': True, 'alive': True, 'dead_status': '9.9.9.9:80: dead'}

In [25]:
# Phase 2 / 3 — teardown: kill the smoke session + stop the fixture (lane
# leaves nothing listening), per the Phase-1 discipline.
import importlib
import dabooks.primitives.sessions as sess
importlib.reload(sess)
st = sess.get_store()
print("sessions:", st.keys())
killed = await st.kill_all()
await st.stop_reaper()
await runner2.cleanup()
print(f"killed: {killed}; remaining: {sess.get_store().keys()}")

sessions: []
killed: []; remaining: []


## Phase 2 gotchas (all live, all fixed)

- **Class-body NameError** — a default like `view_width = Int(view_width)`
  inside the widget class makes the name class-local (class scopes don't
  fall back to the enclosing function for assigned names); the widget now
  lives at module level (`_CockpitWidget`) with per-instance values via
  constructor kwargs.
- **Coordinate scaling** — Phase 1 now pins `deviceScaleFactor=1` (spawn)
  and the view carries `viewport_w/h` traits, so widget-pixel → viewport is
  a plain ratio with no guessing (brief: "mind screenshot scale factor").
- **Escape hatch for button/mouse primitives** — `store.raw(host, port,
  lambda page: …)` runs caller-supplied page lambdas under the same action
  bound; the view never reaches into private page internals.
- **Modifier chords** — the key relay presses held modifiers down, sends
  the key, then releases in reverse (Ctrl-a works).
- **No sync sleeps, no wedges** — failed actions set the error card trait;
  refresh failure keeps the LAST screenshot + marks the view dead (the
  "session ended" card comes from the widget JS, drawn from traits).

## Phase 2 verdict

Green in both lanes: pytest **72/72** (phases 0–2), live loopback smoke
(real chromium → widget state → post-click screenshot refreshed in the
widget: 19.8 KB), dead-view card, and plain-mode fallback verified.
Notebook output above carries the live widget — clicking it forwards
click/drag/keys/scroll into the session (comm relay active).

**STOP for review before Phase 2B** (rewriting proxy + inspector agent —
aiohttp already in the venv).